In [2]:
import os
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    os.chdir('..')
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Taiwei\RestaurantBookings


In [3]:
import pandas as pd
import duckdb

In [4]:
orders = pd.read_csv('./data/interim/train_silver.csv')
restaurants = pd.read_csv('./data/interim/restaurant_silver.csv')
members = pd.read_csv('./data/interim/member_silver.csv')

In [ ]:
restaurants.head()

如果忘記git lfs pull會像下面這樣
```
Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')
```

In [5]:
joined_sql = """
    SELECT
        o.booking_id,
        o.member_id,
        o.cdate,
        o.restaurant_id,
        o.datetime,
        o.people,
        o.purpose,
        o.gender,
        o.status,
        o.is_required_prepay_satisfied,
        o.return90,
        r.is_hotel,
        r.city,
        r.cityarea,
        r.good_for_family,
        r.accept_credit_card,
        r.parking,
        r.outdoor_seating,
        r.wifi,
        r.wheelchair_accessible,
        r.price1,
        r.price2,
        r.lat,
        r.lng,
        m.is_vip,
        m.gender AS member_gender,
        m.birthdate,
        m.city AS member_city
    FROM orders o
    LEFT JOIN restaurants r ON o.restaurant_id = r.id
    LEFT JOIN members m ON o.member_id = m.id
"""

In [6]:
joined_df = duckdb.query(joined_sql).to_df()

In [ ]:
joined_df.to_csv('./data/interim/joined_data.csv', index=False)

In [7]:
joined_df.head()

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,...,wifi,wheelchair_accessible,price1,price2,lat,lng,is_vip,member_gender,birthdate,member_city
0,135323,cef51ca2d0b1b023805351e1ff639fe8634a8fac,2013-09-16 23:57:00,7e8ae24867f137fd20312f43fca2df2fa3b522a1,2013-10-25 19:30:00,2,生日慶祝,F,ok,1,...,1,0,1215,2277,25.08,121.55,0,F,1979-10-10,新北市
1,122342,bb381d78b506f784b618636fca4c91fac246b7f6,2013-06-21 08:39:00,c16962b01ed3e700cd54bc650814a3d1b26261dd,2013-06-30 12:00:00,8,家人用餐,M,canceled,1,...,1,1,726,1305,25.09,121.56,0,M,None,台北市
2,96153,934dd66319005f018a80c461d8e1b4c138a441c7,2013-05-03 12:30:00,7ff292acab039a80acdf847a9d2a91a3432a2deb,2013-05-13 19:30:00,5,家人用餐,F,ok,1,...,1,0,322,501,22.62,120.30,0,M,None,高雄市
3,127578,c2e561f076ee5ce5b49049384ddc48e654126469,2013-05-15 22:49:00,f3837c4489cc6c4587dfb8a92a060585c182ccc1,2013-06-09 12:00:00,2,生日慶祝,F,ok,1,...,1,1,713,976,25.05,121.52,0,M,1993-06-09,台北市
4,22736,22b251b61c5bcb1e84abbd967e9f8fa3f714fcb3,2014-06-23 17:15:00,bb974967d1f63edbd841a954413027e9d9bf4903,2014-06-25 18:00:00,1,Unknown,M,ok,1,...,1,1,880,1100,24.82,121.03,0,M,None,Unknown


In [ ]:
joined_df.dtypes

In [ ]:
members.columns

In [8]:
from src.etl.utils import get_duplicates
dup_df = get_duplicates(members, subset=['is_vip', 'gender', 'birthdate', 'city', 'has_google_id',
       'has_yahoo_id', 'has_weibo_id'])
dup_df.shape

(644686, 9)

In [15]:
from src.etl.utils import get_unique
members_unique = get_unique(members, subset = ['is_vip', 'gender', 'birthdate', 'city', 'has_google_id',
       'has_yahoo_id', 'has_weibo_id','cdate'])
members_unique.shape

(645056, 9)

In [16]:
members.shape

(699199, 9)

In [ ]:
orders.shape

In [6]:
restaurants.shape

(724, 23)

In [7]:
restaurants.head()

,id,is_hotel,country,currency,city,cityarea,name,abbr,tel,opening_hours,...,outdoor_seating,wifi,wheelchair_accessible,price1,price2,lat,lng,timezone,locale,cdate
0,fb1171550a7ad65f5533d24345e7677c5353168d,1,tw,TWD,新北市,Unknown,db231778a1d2a260b510e4379dfb159a63e90e95,e0199e43c23c7322f91049f1c49a9c8d87b3bc39,50f5512c6ed9d5e1b8112e7e52fa3d2aa76514ea,24小時營業(湯屋每次為雙人使用，加時與加人依現場報價為準),...,0,1,1,1100,2200,25.20,121.60,Asia/Taipei,zh_TW,2013-02-27 18:25:00
1,8239c2381e09ba5a298eb9ebefa88de9976105a5,0,tw,TWD,新北市,Unknown,da4531bc68b42dff77fffc7f1b748bfd2a1855b7,da4531bc68b42dff77fffc7f1b748bfd2a1855b7,44d48f31e6b7aa2184018b7d0156fd8e068e3368,週二至週日\r\n午餐：11:00~14:00 \r\n晚餐：17:00~ 21:00\r\...,...,0,1,0,300,400,25.00,121.52,Asia/Taipei,zh_TW,2014-08-26 13:37:00
2,85709eabedeb5c761b424baa16db3bccd6144fd4,1,tw,TWD,台中市,Unknown,e1bb633022c44a15d15a072e17274887707e5774,e1bb633022c44a15d15a072e17274887707e5774,a43a6b1ab17d261040e844ad4db619924c58d188,營業時間：06:30~23:30\r\n\r\n,...,0,1,1,563,736,24.23,120.94,Asia/Taipei,zh_TW,2012-02-25 20:47:00
3,8a035d8ccfaae36efff1cdcc6f02f318f35abd64,1,tw,TWD,台中市,烏日區,2fb3f01285d5d64069513e2f2a06e322e352dcdc,2fb3f01285d5d64069513e2f2a06e322e352dcdc,353015f4db1660c7060c3e9a3063097aff22efa2,平日早餐：6:30- 10:00; \r\n午餐：11:30- 14:00（14:00清場)...,...,0,1,1,379,564,24.14,120.59,Asia/Taipei,zh_TW,2012-06-25 11:59:00
4,56a3a7c9675c5aca5e98ea3d9aa77610ec04183a,0,tw,TWD,台中市,Unknown,408bce81d566a17affd9b56344e0dfd111c69548,408bce81d566a17affd9b56344e0dfd111c69548,caeaa8587bfb7b04cc0d6c1fcf15099decdba1cd,午餐:11:30~14:00\r\n午茶:14:30~16:30\r\n晚餐:17:30~2...,...,0,1,0,499,1000,24.23,120.94,Asia/Taipei,zh_TW,2013-08-15 16:56:00


In [ ]:
members.haed()

,id,is_vip,gender,birthdate,city,has_google_id,has_yahoo_id,has_weibo_id,cdate
0,e732bcf914fecc5a20adc8940689e701160772c4,0,F,1981-05-14,台北市,0,0,0,2012-04-01 00:01:00
1,c671d1bd0b03d682d590275f929b613426dc0e34,0,M,1975-03-22,台北市,0,0,0,2012-01-01 01:22:00
2,fe65711316d561f420bfa3364edf60f0c1859d1f,0,F,1976-11-01,Unknown,0,0,0,2012-01-01 01:32:00
3,6113c28cfc2cf86ae268657683eca4953df54342,0,M,NaN,Unknown,0,0,0,2012-01-01 01:33:00
4,2fc08e00a891d491c99323aed57c4cb53922c959,0,F,1977-01-01,Unknown,0,0,0,2012-01-01 01:54:00


In [12]:
members['birthdate'].value_counts().shape

(17149,)

In [11]:
members['city'].value_counts().shape

(26,)

In [13]:
members['cdate'].value_counts().shape

(503087,)

In [14]:
members.shape

(699199, 9)